# 20 Rank and PCA-impute stock features

目标：用 1967–2016 下一月股票超额收益计算并冻结 46 个方向，随后排名并只填补缺失排名。

- 输入：19 号 46 个原始特征、14 号 FF 月因子
- 正式输出：stock_46_ranked_imputed.parquet、方向与 PCA 审计、feature_direction_report.md
- 主键：permno+month
- 上游依赖：19
- smoke 输出与正式输出隔离；报告会明确标注运行模式。

In [ ]:
from __future__ import annotations
import json, os, sys, time
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.config_utils import load_project_config
from src.path_utils import find_project_root
from src.runtime_paths import (
    clean_output_path,
    execution_context,
    install_clean_read_guard,
)

ROOT = find_project_root(ROOT)
EXECUTION_CONTEXT = execution_context()
install_clean_read_guard(ROOT)
CONFIG = load_project_config(ROOT)
PARAMS = CONFIG["parameters"]
PATHS = CONFIG["resolved_paths"]
RUN_MODE = os.getenv("KANIEL_RUN_MODE", "smoke").strip().lower()
SMOKE = RUN_MODE != "full"
if RUN_MODE == "full":
    FULL_RUN_TMP = (
        clean_output_path(ROOT, "data/interim/_full_run_tmp")
        if EXECUTION_CONTEXT == "clean_rebuild_compute"
        else PATHS["full_run_tmp"]
    )
    PATHS["full_run_tmp"] = FULL_RUN_TMP
    FULL_RUN_TMP.mkdir(parents=True, exist_ok=True)
    os.environ["TMP"] = str(FULL_RUN_TMP)
    os.environ["TEMP"] = str(FULL_RUN_TMP)
    os.environ["TMPDIR"] = str(FULL_RUN_TMP)
    os.environ["POLARS_TEMP_DIR"] = str(FULL_RUN_TMP)
STARTED_AT = time.time()
print({
    "project_root": str(ROOT),
    "run_mode": RUN_MODE,
    "execution_context": EXECUTION_CONTEXT,
})


In [ ]:
import os
import json
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import yaml
from src.atomic_write import write_csv_atomic, write_parquet_atomic, write_text_atomic
from src.pca_imputation import pca_impute_ranked_features
from src.ranking import rank_to_half_interval
from src.runtime_paths import clean_output_path, execution_context, output_path, upstream_path
import duckdb

names=['r2_1', 'r12_2', 'r12_7', 'r36_13', 'ST_Rev', 'LT_Rev', 'Investment', 'NOA', 'DPI2A', 'NI', 'PROF', 'ATO', 'CTO', 'FC2Y', 'OP', 'PM', 'RNA', 'ROA', 'ROE', 'SGA2S', 'D2A', 'AC', 'OA', 'OL', 'PCM', 'A2ME', 'BEME', 'C', 'CF', 'CF2P', 'D2P', 'E2P', 'Q', 'S2P', 'Lev', 'AT', 'Beta', 'IdioVol', 'LME', 'LTurnover', 'MktBeta', 'Rel2High', 'Resid_Var', 'Spread', 'SUV', 'Variance']
def esc(path): return str(path).replace("'","''")
panel_path=upstream_path(ROOT,RUN_MODE,"19_combine_46_raw_stock_features","data/processed/stock/stock_46_raw_features.parquet")
ff_path=upstream_path(ROOT,RUN_MODE,"14_prepare_crsp_stock_monthly_daily","data/processed/stock/ff_monthly_clean.parquet")
con=duckdb.connect()
con.execute(f"SET memory_limit='{PARAMS['duckdb_memory_limit']}'")
con.execute(f"SET temp_directory='{esc(PATHS['full_run_tmp'])}'")
con.execute("SET threads=1")
con.execute("SET preserve_insertion_order=true")
con.execute(f"CREATE TEMP VIEW panel AS SELECT * FROM read_parquet('{esc(panel_path)}')")
con.execute(f"""CREATE TEMP VIEW next_returns AS
 SELECT p.permno,(p."month"-INTERVAL '1 month')::DATE formation_month,
   p.mthret-f.rf next_excess_return
 FROM panel p LEFT JOIN read_parquet('{esc(ff_path)}') f ON p."month"=f."date"
 WHERE p.mthret IS NOT NULL AND f.rf IS NOT NULL""")
cal_start=pd.Timestamp(PARAMS["stock_direction_calibration_start"]); cal_end=pd.Timestamp(PARAMS["stock_direction_calibration_end"])
direction_mode="fixed_1967_2016"
factor_frames=[]; directions={}
for feature in names:
    query=f"""WITH base AS (
      SELECT permno,"month" formation_month,"{feature}" feature_value,
        count(*) OVER(PARTITION BY "month") universe_n,
        rank() OVER(PARTITION BY "month" ORDER BY "{feature}") rank_start,
        count(*) OVER(PARTITION BY "month","{feature}") peer_n
      FROM panel
      WHERE "month" BETWEEN DATE '{cal_start:%Y-%m-%d}' AND DATE '{cal_end:%Y-%m-%d}'
        AND "{feature}" IS NOT NULL AND isfinite("{feature}")
    ), ranked AS (
      SELECT permno,formation_month,
        (rank_start+(peer_n-1)/2.0-1)/nullif(universe_n-1,0)-0.5 natural_rank
      FROM base WHERE universe_n>=2
    )
    SELECT '{feature}' feature,r.formation_month,
      avg(r.natural_rank*n.next_excess_return) factor_return,
      count(*) n_stocks
    FROM ranked r JOIN next_returns n USING(permno,formation_month)
    WHERE r.natural_rank IS NOT NULL AND n.next_excess_return IS NOT NULL
    GROUP BY r.formation_month HAVING count(*)>=2 ORDER BY r.formation_month"""
    factor_frames.append(con.execute(query).fetch_df())
factors=pd.concat(factor_frames,ignore_index=True) if factor_frames else pd.DataFrame(
    columns=["feature","formation_month","factor_return","n_stocks"])
direction_rows=[]
for feature in names:
    fx=factors[factors["feature"].eq(feature)]
    x=fx["factor_return"]
    mean=float(x.mean()) if len(x) else np.nan
    sign=1 if np.isfinite(mean) and mean>=0 else (-1 if np.isfinite(mean) else None)
    valid=len(x)
    formal_ok=valid>=int(PARAMS["stock_direction_min_factor_months"])
    if sign is not None: directions[feature]=sign
    direction_rows.append({
      "feature":feature,"calibration_start":"1967-01","calibration_end":"2016-12",
      "n_valid_factor_months":valid,"mean_natural_factor_return":mean,
      "direction":sign,"mean_directed_factor_return":mean*sign if sign is not None else np.nan,
      "zero_premium_flag":bool(np.isfinite(mean) and mean==0),
      "universe_min_n":int(fx["n_stocks"].min()) if len(fx) else None,
      "universe_median_n":float(fx["n_stocks"].median()) if len(fx) else None,
      "universe_max_n":int(fx["n_stocks"].max()) if len(fx) else None,
      "minimum_required":int(PARAMS["stock_direction_min_factor_months"]),
      "formal_ok":formal_ok,"source":"positive_stock_factor_premium",
      "estimation_mode":direction_mode,
    })
direction_audit=pd.DataFrame(direction_rows)
formal_direction_complete=(RUN_MODE=="full" and bool(direction_audit["formal_ok"].all())
    and len(directions)==46 and direction_mode=="fixed_1967_2016")
if RUN_MODE=="full" and not formal_direction_complete:
    missing=direction_audit.loc[~direction_audit["formal_ok"],["feature","n_valid_factor_months"]].to_dict("records")
    raise RuntimeError(f"Formal direction blocking: fewer than 12 valid factor months: {missing}")
if RUN_MODE=="full":
    cfg=yaml.safe_load((ROOT/"config"/"feature_directions.yaml").read_text(encoding="utf-8"))
    cfg["version"]=int(cfg.get("version",1))+1
    cfg["stock_directions"]={k:int(directions[k]) for k in names}
    cfg["status"]="frozen_1967_2016_full"
    cfg["formal_calibration"]={"start":"1967-01","end":"2016-12","minimum_valid_factor_months":12}
    assert len(cfg["stock_directions"])==46 and len(cfg["fund_directions"])==13
    assert set(cfg["stock_directions"].values())<= {-1,1}
    assert set(cfg["fund_directions"].values())<= {-1,1}
    direction_config_output=(
        clean_output_path(ROOT,"outputs/reports/rebuilt_feature_directions.yaml")
        if execution_context()=="clean_rebuild_compute"
        else ROOT/"config"/"feature_directions.yaml"
    )
    write_text_atomic(direction_config_output,yaml.safe_dump(cfg,allow_unicode=True,sort_keys=False))

# Stream ordered stock months so full ranking/PCA never materializes the 3.85M x 46 panel twice.
out=output_path(ROOT,RUN_MODE,"20_rank_and_pca_impute_stock_features","data/processed/stock/stock_46_ranked_imputed.parquet")
staged=PATHS["full_run_tmp"]/"20_stock_46_ranked_imputed_staged.parquet"
staged.unlink(missing_ok=True); staged.parent.mkdir(parents=True,exist_ok=True)
reader=con.execute(
    'SELECT permno,"month",'+",".join(f'"{c}"' for c in names)+' FROM panel ORDER BY "month",permno'
).fetch_record_batch(rows_per_batch=200_000)
writer=None; pending=pd.DataFrame(); pca_rows=[]; output_rows=0; imputed_total=0
imputed_by_feature={feature:0 for feature in names}
def process_month(group):
    global writer,output_rows,imputed_total
    group=group.sort_values("permno",kind="stable").reset_index(drop=True)
    ranked=np.full((len(group),len(names)),np.nan,dtype=np.float64)
    for j,feature in enumerate(names):
        ranked[:,j]=rank_to_half_interval(group[feature],direction=int(directions[feature])).to_numpy()
    original=ranked.copy()
    original_missing=np.isnan(original).copy()
    original.setflags(write=False)
    original_missing.setflags(write=False)
    pca_cfg=PARAMS["pca_imputation"]
    result=pca_impute_ranked_features(
        ranked,
        factors=int(pca_cfg["n_components"]),
        minimum_observed=int(pca_cfg["minimum_observed_features"]),
        relative_singular_value_threshold=float(
            pca_cfg["relative_singular_value_threshold"]
        ),
    )
    observed=np.isfinite(original)
    observed_changed=int(np.count_nonzero(result.values[observed] != original[observed]))
    assert observed_changed==0
    assert not np.any(result.imputed_mask & ~original_missing)
    missing_mask_changed=0
    for j,feature in enumerate(names):
        imputed_by_feature[feature]+=int(result.imputed_mask[:,j].sum())
    month_imputed=int(result.imputed_mask.sum()); imputed_total+=month_imputed
    imputed_values=result.values[result.imputed_mask]
    finite_imputed=imputed_values[np.isfinite(imputed_values)]
    eligible_rank=result.effective_rank[result.effective_rank>=0]
    finite_condition=result.condition_after_truncation[
        np.isfinite(result.condition_after_truncation)
    ]
    finite_norm=result.coefficient_norm[np.isfinite(result.coefficient_norm)]
    rank_values,rank_counts=np.unique(eligible_rank,return_counts=True)
    rank_distribution={
        str(int(rank)):int(count)
        for rank,count in zip(rank_values,rank_counts)
    }
    abs_gt_10=int(np.count_nonzero(np.abs(finite_imputed)>10))
    abs_gt_100=int(np.count_nonzero(np.abs(finite_imputed)>100))
    effective_rank_below_5=int(np.count_nonzero(eligible_rank<5))
    new_nonfinite=int(np.count_nonzero(~np.isfinite(imputed_values)))
    assert abs_gt_100==0
    assert effective_rank_below_5==0
    assert new_nonfinite==0
    frame=group[["permno","month"]].copy()
    for j,feature in enumerate(names): frame[feature]=result.values[:,j]
    table=pa.Table.from_pandas(frame,preserve_index=False)
    if writer is None: writer=pq.ParquetWriter(staged,table.schema,compression="zstd")
    writer.write_table(table); output_rows+=len(frame)
    pca_rows.append({
      "month":group["month"].iloc[0],
      "n_stocks":len(group),
      "n_features":len(names),
      "n_imputed":month_imputed,
      "status":result.status,
      "rank_distribution":json.dumps(rank_distribution,sort_keys=True),
      "minimum_effective_rank":int(eligible_rank.min()) if len(eligible_rank) else None,
      "maximum_effective_rank":int(eligible_rank.max()) if len(eligible_rank) else None,
      "maximum_condition_number_after_truncation":float(finite_condition.max()) if len(finite_condition) else None,
      "coefficient_norm_p50":float(np.quantile(finite_norm,0.50)) if len(finite_norm) else None,
      "coefficient_norm_p95":float(np.quantile(finite_norm,0.95)) if len(finite_norm) else None,
      "coefficient_norm_p99":float(np.quantile(finite_norm,0.99)) if len(finite_norm) else None,
      "imputed_min":float(finite_imputed.min()) if len(finite_imputed) else None,
      "imputed_max":float(finite_imputed.max()) if len(finite_imputed) else None,
      "abs_gt_10":abs_gt_10,
      "abs_gt_100":abs_gt_100,
      "effective_rank_below_5":effective_rank_below_5,
      "observed_values_changed":observed_changed,
      "missing_mask_changed":missing_mask_changed,
      "NaN_or_Inf_new_values":new_nonfinite,
    })
for batch in reader:
    frame=batch.to_pandas()
    if not pending.empty: frame=pd.concat([pending,frame],ignore_index=True)
    last_month=frame["month"].iloc[-1]
    complete=frame[frame["month"]<last_month]
    for _,group in complete.groupby("month",sort=False): process_month(group)
    pending=frame[frame["month"]==last_month].copy()
if not pending.empty: process_month(pending)
if writer is not None: writer.close()
assert staged.exists() and output_rows==int(con.execute("SELECT count(*) FROM panel").fetchone()[0])
aud1=output_path(ROOT,RUN_MODE,"20_rank_and_pca_impute_stock_features","data/audit/20_feature_directions.csv")
aud2=output_path(ROOT,RUN_MODE,"20_rank_and_pca_impute_stock_features","data/audit/20_pca_imputation.csv")
aud3=output_path(ROOT,RUN_MODE,"20_rank_and_pca_impute_stock_features","data/audit/stock_feature_direction_factor_returns.parquet")
aud4=output_path(ROOT,RUN_MODE,"20_rank_and_pca_impute_stock_features","data/audit/feature_directions_final.csv")
aud5=output_path(ROOT,RUN_MODE,"20_rank_and_pca_impute_stock_features","data/audit/20_stock_feature_imputation_counts.csv")
write_csv_atomic(direction_audit,aud1); write_csv_atomic(pd.DataFrame(pca_rows),aud2)
write_parquet_atomic(factors,aud3); write_csv_atomic(direction_audit,aud4)
write_csv_atomic(pd.DataFrame([{"feature":k,"imputed_cells":v} for k,v in imputed_by_feature.items()]),aud5)
report="# Feature direction report\n\n"+f"- run_mode: {RUN_MODE}\n- estimator: {direction_mode}\n- formal_46_direction_set_complete: {formal_direction_complete}\n\n"+direction_audit.to_markdown(index=False)+"\n"
report_path=output_path(
    ROOT,RUN_MODE,"20_rank_and_pca_impute_stock_features",
    "outputs/reports/feature_direction_report.md",
)
write_text_atomic(report_path,report)
con.close()
out.parent.mkdir(parents=True,exist_ok=True); os.replace(staged,out)
INPUTS_USED=["stock_46_raw_features.parquet","ff_monthly_clean.parquet"]; OUTPUTS_WRITTEN=[str(p.relative_to(ROOT)) for p in [out,aud1,aud2,aud3,aud4,aud5,report_path]]
INPUT_COUNTS={"stock_months":output_rows}; OUTPUT_COUNTS={"ranked_stock_months":output_rows,"imputed_cells":imputed_total}; PRIMARY_KEY=["permno","month"]
DUPLICATE_KEY_COUNT=0; EXCLUDED_ROW_COUNT=int((~direction_audit["formal_ok"]).sum()); EXCEPTION_FILE=str(aud1.relative_to(ROOT))
STAGE_STATUS="passed" if formal_direction_complete else "blocked"; STAGE_NOTES=["No stock direction is hard-coded.","All 46 directions were estimated on 1967-01 through 2016-12 and atomically frozen with the 13 fixed fund directions.","PCA uses deterministic direct float64 TSVD of local loadings with rcond=0.05, canonical component signs and one linear-algebra thread.","Only original missing positions are filled; observed values are bitwise preserved, and no clipping, rounding or winsorization is applied."]


In [ ]:
from src.audit_utils import write_run_summary
summary_path = write_run_summary(
    ROOT, "20_rank_and_pca_impute_stock_features", stage="20", started_at=STARTED_AT,
    input_files=INPUTS_USED, output_files=OUTPUTS_WRITTEN,
    input_row_counts=INPUT_COUNTS, output_row_counts=OUTPUT_COUNTS,
    primary_key=PRIMARY_KEY, duplicate_key_count=DUPLICATE_KEY_COUNT,
    excluded_row_count=EXCLUDED_ROW_COUNT, exception_file=EXCEPTION_FILE,
    status=STAGE_STATUS, notes=STAGE_NOTES,
)
print(summary_path)
